In [0]:
# Notebook: landing_to_bronze
# Author: Virendra Dilip Tambavekar
# HRM 6217
# Task: Load 'cross' domain data from landing to bronze layer using Databricks utilities

In [0]:
#Importing required libraries
import logging
from pyspark.sql.functions import *
from pyspark.sql.utils import AnalysisException
#Initialize logger
logger = logging.getLogger("LandingToBronze_CrossDomin")
logger.setLevel(logging.INFO)

In [0]:
# Parameterization
dbutils.widgets.text("batch_id","1")

batch_id = dbutils.widgets.get("batch_id")

# Base Paths & Schemas
landing_base_path = f"/Volumes/charles_schwab_retailbrokerage_dev_team_lemma/landing/pwg/Batch{batch_id}/"
bronze_schema = f"charles_schwab_retailbrokerage_dev_team_lemma.bronze"

#List of Cross-Domain Files in Batch 1
cross_domain_files = [
    "Date","Time","StatusType","TaxRate","Industry","TradeType"
]

In [0]:
def bronze_ingestion(file_name: str) -> dict:
    source_path = f"{landing_base_path}{file_name}"
    target_table = f"{bronze_schema}.{file_name.lower()}"

    logger.info(f"Processing file {file_name} for Bronze ingestion")
    try:
        #Read file
        landing_df = spark.read.parquet(source_path)
        landing_count = landing_df.count()
        #Metadata
        bronze_df = (
            landing_df
            .withColumn("_ingest_ts",current_timestamp())
            .drop("_landing_ts")
        )
        #Write to bronze table
        (
            bronze_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .partitionBy("_batch")
            .saveAsTable(target_table)
        )
        #Reconciliation validation
        written_df = spark.table(target_table).filter(col("_batch") == batch_id)
        bronze_count = written_df.count()
        status = "SUCCESS" if landing_count == bronze_count else "COUNT_MISMATCH"

        return {
            "Entity" : file_name,
            "Landing_Count" : landing_count,
            "Bronze_Count" : bronze_count,
            "Status" : status
        }
    except AnalysisException as e:
        if "Path does not exist" in str(e):
            logger.info(f"{file_name} not found in Landing batch for Batch{batch_id}")
            return {
                "Entity" : file_name,
                "Landing_Count" : 0,
                "Bronze_Count" : 0,
                "Status" : "NOT FOUND"
            } 
        else:
            logger.error(f"Error Processing {file_name}")
            raise e

In [0]:
def main():
    logger.info(f"Starting to ingestion in Bronze Layer | Batch: {batch_id}")

    reconciliation_results = []
    for file_name in cross_domain_files:
        metrics = bronze_ingestion(file_name)
        reconciliation_results.append(metrics)

    logger.info("Pipeline Completed")
    recon_df = spark.createDataFrame(reconciliation_results)

    mismatches = recon_df.filter(col("STATUS") == "COUNT_MISMATCH").count()
    if mismatches > 0:
        logger.warning(f"Reconciliation failed for {mismatches} files")

    display(recon_df)

main()